# The Price Is Right - Extension

This notebook enhances week 8 project by integrating an additional agent that will send a message to another person e.g. *your brother*, to review the price and if they agree with the discount, the agent will further send a purchase order request to the merchant.

In [ ]:
# Imports
import sys
import os
from typing import override, List, Optional

# Change working directory to incorporate week 8 project.
notebook_dir = os.getcwd()
week8_dir = os.path.join(notebook_dir, '..', '..')
os.chdir(week8_dir)
print(f"Working directory: {os.getcwd()}")

import logging

### Price Review Agent
Will send a message to a different person to review the discount. If they agree with the discount offered as analysed by the LLM, the agent will then proceed to send a purchase order to the merchant.

In [ ]:
from agents.agent import Agent
from agents.deals import Deal, Opportunity
from litellm import completion
import requests

pushover_url = "https://api.pushover.net/1/messages.json"

class PriceReviewAgent(Agent):
    name = "Price Review Agent"
    color = Agent.RED
    MODEL = "claude-sonnet-4-5"
    REVIEW_RESPONSE = "I agree with the price"

    def __init__(self):
        """
        This agent will do push notifications via Pushover,
        """
        self.log(f"{self.name} is initializing")
        self.pushover_user = os.getenv("PUSHOVER_USER", "your-pushover-user-if-not-using-env")
        self.pushover_token = os.getenv("PUSHOVER_TOKEN", "your-pushover-user-if-not-using-env")
        self.log(f"{self.name} has initialized Pushover and Claude")
    
    def push(self, text):
        """
        Send a Push Notification using the Pushover API
        """
        self.log(f"{self.name} is sending a push notification")
        payload = {
            "user": self.pushover_user,
            "token": self.pushover_token,
            "message": text,
            "sound": "cashregister",
        }
        requests.post(pushover_url, data=payload)

    def craft_review_request_message(
        self, description: str, deal_price: float, estimated_true_value: float
    ) -> str:
        """
        Craft a review request message to your brother to review the offer and if they agree with the discount.
        """
        self.log(f"{self.name} is crafting a review request message for the deal")
        user_prompt = "Please summarize the following deal in 4-5 sentences to be sent as a review request message to my brother. They should reply to the message with their approval or disapproval.\n"
        user_prompt += f"Item Description: {description}\nOffered Price: {deal_price}\nEstimated true value: {estimated_true_value}"
        user_prompt += "\n\nRespond only with the 4-5 sentence message which will be sent to my brother asking them to review the offer."
        response = completion(
            model=self.MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        return response.choices[0].message.content
    
    def send_review_request(self, opportunity: Opportunity):
        """
        Send a review request message.
        """
        self.log(f"{self.name} is sending a review request message")
        message = self.craft_review_request_message(
            opportunity.deal.product_description, opportunity.deal.price, opportunity.estimate)
        self.push(message)

    def get_review_response(self) -> str:
        """
        Get the review response from the reveiwer.
        """
        self.log(f"{self.name} is getting the review response")
        return self.REVIEW_RESPONSE
    
    def analyse_review_response(self, message: str) -> bool:
        """
        Analyse the review response and return a boolean value
        """
        self.log(f"{self.name} is analysing the review response")
        user_prompt = f"Please analyse the following message from the reveiwer and respond with 'true' indicating if they agree with the discount. Respond with an empty string if they do not agree with the discount.\n"
        user_prompt += f"Message: {message}"
        response = completion(
            model=self.MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        return True if response.choices[0].message.content == "true" else False

    def craft_purchase_order(self, opportunity: Opportunity):
        """
        Send a purchase order for the deal
        """
        self.log(f"{self.name} is sending a purchase order for the deal to the merchant.")
        user_prompt = f"Please craft a purchase order for the following deal to be sent to the merchant.\n"
        user_prompt += f"Deal: {opportunity.deal.product_description}\nOffered Price: {opportunity.deal.price}\nEstimated true value: {opportunity.estimate}"
        response = completion(
            model=self.MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        return response.choices[0].message.content
    
    def send_purchase_order(self, opportunity: Opportunity):
        """
        Send a purchase order for the deal
        """
        self.log(f"{self.name} is sending a purchase order for the deal to the merchant.")
        message = self.craft_purchase_order(opportunity)
        self.push(message)
    
    

### Extended Planning Agent
Add the price review agent functionality to the Planning agent.

In [ ]:
from agents.planning_agent import PlanningAgent

class ExtendedPlanningAgent(PlanningAgent):
    def __init__(self, collection):
        super().__init__(collection)
        self.review_agent = PriceReviewAgent()

    @override
    def plan(self, memory: List[str] = []) -> Optional[Opportunity]:
        """
        Run the full workflow:
        1. Use the ScannerAgent to find deals from RSS feeds
        2. Use the EnsembleAgent to estimate them
        3. Use the MessagingAgent to send a notification of deals
        4. Use the PriceReviewAgent to send a review request to the brother and send a purchase order if they agree with the discount
        """
        self.log("Planning Agent is kicking off a run")
        selection = self.scanner.scan(memory=memory)
        if selection:
            opportunities = [self.run(deal) for deal in selection.deals[:5]]
            opportunities.sort(key=lambda opp: opp.discount, reverse=True)
            best = opportunities[0]
            self.log(f"Planning Agent has identified the best deal has discount ${best.discount:.2f}")
            if best.discount > self.DEAL_THRESHOLD:
                self.messenger.alert(best)
                self.review_agent.send_review_request(best)
                review_response = self.review_agent.get_review_response()
                self.review_agent.analyse_review_response(review_response)
                if self.review_agent.analyse_review_response(review_response):
                    self.review_agent.send_purchase_order(best)
            self.log("Planning Agent has completed a run")
            return best if best.discount > self.DEAL_THRESHOLD else None
        return None


### Final Agent Framework
Add the Extended Planning Agent to the application.

In [ ]:
from deal_agent_framework import DealAgentFramework

class ExtendedDealAgentFramework(DealAgentFramework):
    def __init__(self):
        super().__init__()
    
    @override
    def init_agents_as_needed(self):
        if not self.planner:
            self.log("Initializing Agent Framework")
            self.planner = ExtendedPlanningAgent(self.collection)
            self.log("Agent Framework is ready")


### Run the final system of agents.

In [ ]:
app = ExtendedDealAgentFramework()
app.run()